# 🤖 Section 1: What is an AI Agent?

You've probably used an LLM like Gemini before, where you give it a prompt and it gives you a text response.

`Prompt -> LLM -> Text`

An AI Agent takes this one step further. An agent can think, take actions, and observe the results of those actions to give you a better answer.

`Prompt -> Agent -> Thought -> Action -> Observation -> Final Answer`

We'll configure an `Agent` by setting its key properties, which tell it what to do and how to operate.

These are the main properties we'll set:

- **name** and **description**: A simple name and description to identify our agent.
- **model**: The specific LLM that will power the agent's reasoning. We'll use "gemini-2.5-flash-lite".
- **instruction**: The agent's guiding prompt. This tells the agent its goal is and how to behave.
- **tools**: A list of [tools](https://google.github.io/adk-docs/tools/) that the agent can use. To start, we'll give it the `google_search` tool, which lets it find up-to-date information online.

## 1.1. Why Multi-Agent Systems? 🤔

**The Problem: The "Do-It-All" Agent**

Single agents can do a lot. But what happens when the task gets complex? A single "monolithic" agent that tries to do research, writing, editing, and fact-checking all at once becomes a problem. Its instruction prompt gets long and confusing. It's hard to debug (which part failed?), difficult to maintain, and often produces unreliable results.

**The Solution: A Team of Specialists**

Instead of one "do-it-all" agent, we can build a **multi-agent system**. This is a team of simple, specialized agents that collaborate, just like a real-world team. Each agent has one clear job (e.g., one agent *only* does research, another *only* writes). This makes them easier to build, easier to test, and much more powerful and reliable when working together.

**Architecture: Single Agent vs Multi-Agent Team**

<img width="800" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/multi-agent-team.png" alt="Multi-agent Team" />

# 🎬 Section 2: Memory Management (Session)

At their core, Large Language Models are **inherently stateless**. Their awareness is confined to the information you provide in a single API call. This means an agent without proper context management will react to the current prompt without considering any previous history.

**❓ Why does this matter?** Imagine trying to have a meaningful conversation with someone who forgets everything you've said after each sentence. That's the challenge we face with raw LLMs!

In ADK, we use `Sessions` for **short term memory management** and `Memory` for **long term memory.** In the next notebook, you'll focus on `Memory`.

A session is a container for conversations. It encapsulates the conversation history in a chronological manner and also records all tool interactions and responses for a **single, continuous conversation**. A session is tied to a user and agent; it is not shared with other users. Similarly, a session history for an Agent is not shared with other Agents.

a **Session** is comprised of two key components `Events` and `State`:

**📝 Session.Events**:

> While a session is a container for conversations, `Events` are the building blocks of a conversation.
>
> Example of Events:
> - *User Input*: A message from the user (text, audio, image, etc.)
> - *Agent Response*: The agent's reply to the user
> - *Tool Call*: The agent's decision to use an external tool or API
> - *Tool Output*: The data returned from a tool call, which the agent uses to continue its reasoning
    

**{} Session.State**:

> `session.state` is the Agent's scratchpad, where it stores and updates dynamic details needed during the conversation. Think of it as a global `{key, value}` pair storage which is available to all subagents and tools.

<img src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day3/session-state-and-events.png" width="320" alt="Session state and events">

# 🧠 Section 3: Memory Management (Memory)

### What is Memory ❓

Memory is a service that provides long-term knowledge storage for your agents. The key distinction:

> **Session = Short-term memory** (single conversation)
> 
> **Memory = Long-term knowledge** (across multiple conversations)

Think of it in software engineering terms: **Session** is like application state (temporary), while **Memory** is like a database (persistent).

- 🗣️ **Session**: They remember what you said 10 minutes ago in THIS conversation
- 🧠 **Memory**: They remember your preferences from conversations LAST WEEK

In order to integrate Memory into your Agents, there are **three high-level steps.**

**Three-step integration process:**

1. **Initialize** → Create a `MemoryService` and provide it to your agent via the `Runner`
2. **Ingest** → Transfer session data to memory using `add_session_to_memory()`
3. **Retrieve** → Search stored memories using `search_memory()`

<img src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day3/memory-workflow.png" width="1400" alt="Memory workflow">

# 🎯 AI Career Advisor Multi-Agent System

An intelligent career transition assistant built with (https://google.github.io/adk-docs/)[text] (https://google.github.io/adk-docs/)[Google's Agent Development Kit (ADK)].

This agent helps people navigate career transitions by:
1. **Researching** target career paths (skills, resources, market outlook)
2. **Creating** personalized action plans (3-phase transition roadmap)

## Architecture

The system uses a **Sequential Agent** pattern:
```
User Query → Research Agent → Mentor Agent → Career Plan
```

This agent acts like an assembly line, running each sub-agent in the exact order you list them. The output of one agent automatically becomes the input for the next, creating a predictable and reliable workflow.

**Research Agent**: Uses Google Search to gather information about career transitions
**Mentor Agent**: Creates personalized 3-phase action plans based on research

## Key Features

- ✅ **Memory Integration**: Remembers context across sessions
- ✅ **Structured Output**: Organized research and actionable plans
- ✅ **Automated Callbacks**: Saves conversations to memory automatically
- ✅ **Error Handling**: Graceful retry logic for API failures

## How to Use

1. Run all cells in order
2. Provide a career transition query (include: current role, experience, target role, time commitment)
3. Receive comprehensive research and a personalized action plan

**Example Query:**
> "I am a Product Manager with 5 years of experience. I want to transition to data science and can dedicate 10 hours per week."

---
## 1️⃣ Setup & Configuration

First, we'll set up the environment and load necessary dependencies.

In [1]:
# Import required libraries
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Import ADK components
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.runners import Runner
from google.adk.models.google_llm import Gemini
from google.adk.tools import google_search, preload_memory
from google.genai import types
from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService

print("✅ All imports loaded successfully")

✅ All imports loaded successfully


In [2]:
# Set up Gemini API key from environment variable
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if GOOGLE_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
else:
    print("🔑 Authentication Error: 'GOOGLE_API_KEY' not found in .env file.")

✅ Gemini API key setup complete.


In [3]:
# Application constants
APP_NAME = "career_advisor"
USER_ID = "default"
MODEL_NAME = "gemini-2.5-flash-lite"

In [4]:
async def auto_save_to_memory(callback_context):
    """
    Automatically save session to memory after each agent turn.

    This callback:
    - Extracts the agent name and session from the callback context
    - Saves the session to memory service
    - Logs success or failure
    - Fails gracefully without breaking the agent flow

    Args:
        callback_context: The callback context provided by ADK
    """
    try:
        agent_name = callback_context._invocation_context.agent.name
        session = callback_context._invocation_context.session

        await callback_context._invocation_context.memory_service.add_session_to_memory(session)

        print(f"💾 Saved {agent_name} output to memory")
    except Exception as e:
        print(f"⚠️  Warning: Failed to save to memory: {e}")
        

# Helper function for running sessions
async def run_session(
    runner_instance: Runner,
    user_queries: list[str] | str = None,
    session_name: str = "default",
    show_all_agents: bool = False,
):
    """Run queries in a session and display agent responses."""
    print(f"\n{'='*60}")
    print(f"Session: {session_name}")
    print(f"{'='*60}")

    app_name = runner_instance.app_name

    # Create or retrieve session
    try:
        session = await session_service.create_session(
            app_name=app_name, user_id=USER_ID, session_id=session_name
        )
        print("✅ New session created")
    except:
        session = await session_service.get_session(
            app_name=app_name, user_id=USER_ID, session_id=session_name
        )
        print("✅ Existing session retrieved")

    if user_queries:
        if isinstance(user_queries, str):
            user_queries = [user_queries]

        for query in user_queries:
            print(f"\n👤 User: {query}")
            print(f"{'-'*60}")

            query_content = types.Content(role="user", parts=[types.Part(text=query)])

            responses = []

            async for event in runner_instance.run_async(
                user_id=USER_ID, session_id=session.id, new_message=query_content
            ):
                if event.content and event.content.parts:
                    text = event.content.parts[0].text
                    if text and text != "None":
                        agent_name = getattr(event, 'author', 'Agent')

                        if show_all_agents:
                            print(f"\n🤖 {agent_name}:")
                            print(text)
                        else:
                            responses.append({'agent': agent_name, 'text': text})

            # If not showing all, print only final response
            if not show_all_agents and responses:
                last = responses[-1]
                print(f"\n🤖 {last['agent']}:")
                print(last['text'])
    else:
        print("⚠️  No queries provided!")

print("✅ Helper function defined.")

✅ Helper function defined.


In [5]:
# Retry configuration for API calls
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Exponential backoff multiplier
    initial_delay=1,  # Initial delay in seconds
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

---
## 2️⃣ Agent Definitions

Now we'll define our two specialized agents that work together in sequence.

### Research Agent

The **Research Agent** is responsible for gathering information about career transitions.

**Tools:**
- `google_search`: Searches the web for relevant information
- `preload_memory`: Checks past conversations for context

**Output:** Structured research with:
- Key skills required
- Learning resources
- Market outlook
- Transition timeline
- Success stories

In [ ]:
# 1. Research Agent
research_agent = LlmAgent(
    name="research_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""
    You are a Research Agent specialized in career transitions.
    
    Your task:
    1. Extract career transition details from the user's message or check preload_memory for context
    2. Use the Google Search tool to research the target career path
    3. Focus on: required skills, typical career progression, salary ranges, and job market demand
    4. Search for: online courses, certifications, and learning resources
    5. Look for: success stories of people who made similar transitions
    
    Output format:
    ## Key Skills Required
    [List 5-7 essential skills with brief descriptions]
    
    ## Learning Resources
    [Specific courses, certifications, books, and platforms]
    
    ## Market Outlook
    [Job demand, salary ranges, growth trends with data]
    
    ## Transition Timeline
    [Typical timeframe for this transition based on the user's experience level]
    
    ## Success Stories
    [Brief examples of successful transitions]
    
    Keep your research comprehensive but concise. Focus on actionable information tailored to the user's background.
    """,
    tools=[google_search, preload_memory],
    output_key="research_summary", # The result of this agent will be stored in the session state under the key 'research_summary'
    after_agent_callback=auto_save_to_memory,
)

### Mentor Agent

The **Mentor Agent** creates personalized action plans based on research findings.

**Input:** Research summary from the Research Agent

**Output:** 3-phase transition plan with:
- **Phase 1 (Months 1-3)**: Foundation skills and quick wins
- **Phase 2 (Months 4-6)**: Portfolio building and networking
- **Phase 3 (Months 6-9)**: Job search preparation
- **Milestones**: Monthly goals and progress tracking
- **Leveraging Background**: How to use existing skills

In [7]:
# 2. Mentor Agent
mentor_agent = LlmAgent(
    name="mentor_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""
    You are a Career Mentor Agent. Based on the research findings: {research_summary}
    
    Extract the user's context from the conversation history and research summary, then create a personalized, actionable transition plan.
    
    Structure your plan as follows:
    
    ## Phase 1: Foundation (Months 1-3)
    - Specific skills to learn first (prioritized based on their background)
    - Recommended courses/resources with links when available
    - Daily/weekly time commitment suggestions
    - Quick wins to build confidence
    
    ## Phase 2: Building Portfolio (Months 4-6)
    - Concrete projects to build (with examples relevant to their experience)
    - GitHub repositories to create
    - Communities to join (specific names)
    - Networking strategies leveraging their current role
    
    ## Phase 3: Job Search (Months 6-9)
    - Resume updates needed (specific sections)
    - Where to apply (companies, job boards)
    - Interview preparation tips
    - Portfolio presentation strategies
    
    ## Milestones & Checkpoints
    - Monthly goals with measurable outcomes
    - How to measure progress
    - Red flags and when to adjust course
    
    ## Leveraging Your Background
    - How to translate their existing skills to the new role
    - Unique advantages from their current position
    - How their experience is an asset
    
    Make it specific, realistic, and encouraging. Use actual course names, platforms, and communities when possible.
    Keep the tone supportive but practical - acknowledge challenges while emphasizing achievability.
    Tailor everything to their specific situation.
    """,
    output_key="career_advice",
)

---
## 3️⃣ Pipeline Assembly

Now we'll combine our agents into a sequential pipeline and create the runner.

In [8]:
# 3. Root Agent to orchestrate the workflow
root_agent = SequentialAgent(
    name="CareerPathPipeline",
    sub_agents=[research_agent, mentor_agent],
)

print("✅ Root agent created with 2-agent pipeline: Research -> Mentor")

✅ Root agent created with 2-agent pipeline: Research -> Mentor


### Initialize Runner

The **Runner** orchestrates the execution of our agent pipeline.

**Components:**
- `session_service`: Manages conversation history (short-term memory)
- `memory_service`: Stores knowledge across sessions (long-term memory)

Both agents in the pipeline will have access to these services.

In [ ]:
# Initialize services
session_service = InMemorySessionService()  # Short-term conversation memory (stores conversations in RAM temporarily)
memory_service = InMemoryMemoryService()    # Long-term knowledge storage (Remembers conversations from previous days)

# Create the Runner
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
    memory_service=memory_service,
)

print("✅ Runner initialized successfully!")
print(f"   Application: {APP_NAME}")
print(f"   User: {USER_ID}")
print(f"   Model: {MODEL_NAME}")

App name mismatch detected. The runner is configured with app name "career_advisor", but the root agent was loaded from "/Users/atawua/Documents/03-DATA-SCIENCE/01-github/ai-agent/.venv/lib/python3.11/site-packages/google/adk/agents", which implies app name "agents".


✅ Runner initialized successfully!
   Application: career_advisor
   User: default
   Model: gemini-2.5-flash-lite


---
## 4️⃣ Testing the Agent

Now let's test our career advisor with a sample query.

**What to include in your query:**
- Current role
- Years of experience
- Target role
- Time commitment (hours per week)

The agent will research the career path and create a personalized transition plan.

In [10]:
# Run the career advisor with a sample query
await run_session(
    runner,
    "I am a Product Manager with 5 years of experience. I want to transition to data science and can dedicate 10 hours per week.",
    session_name="career-transition-demo",
)


Session: career-transition-demo
✅ New session created

👤 User: I am a Product Manager with 5 years of experience. I want to transition to data science and can dedicate 10 hours per week.
------------------------------------------------------------
💾 Saved research_agent output to memory

🤖 mentor_agent:
It's fantastic that you're looking to leverage your 5 years of Product Management experience to transition into Data Science! Your background gives you a unique advantage in understanding business needs, user perspectives, and the strategic impact of data. With a dedicated 10 hours per week, a transition to an entry-level Data Scientist role is achievable within **12-18 months**.

Here's a personalized, actionable transition plan designed to build upon your strengths:

## Phase 1: Foundation (Months 1-3)

This phase is about building your core technical toolkit. As a former PM, you're already skilled at understanding problems, which is a huge head start.

*   **Specific Skills to Learn